# 0. import libraries

In [ ]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(26):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-1b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

print(transcoder_paths)  # verify all 26 are present

transcoder_all/layer_0_width_16k_l0_smal(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub\models--google--gemma-scope-2-1b-pt. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


transcoder_all/layer_1_width_16k_l0_smal(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

transcoder_all/layer_2_width_16k_l0_smal(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

transcoder_all/layer_3_width_16k_l0_smal(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

...

transcoder_all/layer_22_width_16k_l0_sma(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

transcoder_all/layer_23_width_16k_l0_sma(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

transcoder_all/layer_24_width_16k_l0_sma(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

transcoder_all/layer_25_width_16k_l0_sma(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

{0: 'D:\\huggingface_cache\\hub\\models--google--gemma-scope-2-1b-pt\\snapshots\\b738dc06961818c011fb2e44a316352ca0f4e873\\transcoder_all\\layer_0_width_16k_l0_small\\params.safetensors', 1: 'D:\\huggingface_cache\\hub\\models--google--gemma-scope-2-1b-pt\\snapshots\\b738dc06961818c011fb2e44a316352ca0f4e873\\transcoder_all\\layer_1_width_16k_l0_small\\params.safetensors', 2: 'D:\\huggingface_cache\\hub\\models--google--gemma-scope-2-1b-pt\\snapshots\\b738dc06961818c011fb2e44a316352ca0f4e873\\transcoder_all\\layer_2_width_16k_l0_small\\params.safetensors', 3: 'D:\\huggingface_cache\\hub\\models--google--gemma-scope-2-1b-pt\\snapshots\\b738dc06961818c011fb2e44a316352ca0f4e873\\transcoder_all\\layer_3_width_16k_l0_small\\params.safetensors', 4: 'D:\\huggingface_cache\\hub\\models--google--gemma-scope-2-1b-pt\\snapshots\\b738dc06961818c011fb2e44a316352ca0f4e873\\transcoder_all\\layer_4_width_16k_l0_small\\params.safetensors', 5: 'D:\\huggingface_cache\\hub\\models--google--gemma-scope-2-1b

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-1b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\circuit_tracer\transcoder\single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [6]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-1b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub\models--google--gemma-3-1b-pt. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-3-1b-pt into HookedTransformer


# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [7]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 1024  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 32  # Batch size when attributing
offload = "cpu"
verbose = True  # Whether to display a tqdm progress bar and timing report

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [8]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,   # ← missing
        batch_size=batch_size,
        offload=offload,                        # ← use the variable, not hardcoded "cpu"
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files/gemma-3-1b",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

Phase 0: Precomputing activations and vectors


Generated: He saw a carrot and had to grab it,
He saw a carrot and had to grab it


Precomputation completed in 2.51s
Found 1289771 active features
Phase 1: Running forward pass
Forward pass completed in 1.42s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6289
Will include 1024 of 1289771 feature nodes
Input vectors built in 3.00s
Phase 3: Computing logit attributions
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\circuit_tracer\attribution\context_transformerlens.py:223: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 41.70s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [13:43<00:00,  1.24it/s]
Feature attributions completed in 823.78s
Attribution completed in 872.

✓ step 00 → 'He'  saved as 'step-00-He'


Precomputation completed in 7.48s
Found 1364301 active features
Phase 1: Running forward pass
Forward pass completed in 1.49s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5547
Will include 1024 of 1364301 feature nodes
Input vectors built in 1.14s
Phase 3: Computing logit attributions
Logit attributions completed in 38.30s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [14:51<00:00,  1.15it/s]
Feature attributions completed in 891.03s
Attribution completed in 939.95s
Phase 0: Precomputing activations and vectors


...

✓ step 17 → ' to'  saved as 'step-17-to'


Precomputation completed in 14.10s
Found 2641473 active features
Phase 1: Running forward pass
Forward pass completed in 7.55s
Phase 2: Building input vectors
Using 1 salient logits with cumulative probability 0.9883
Will include 1024 of 2641473 feature nodes
Input vectors built in 5.00s
Phase 3: Computing logit attributions
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\torch\functional.py:373: UserWarning: mkldnn_matmul failed, switching to baddbmm:[enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 21045141504 bytes.
00007FFA93C451B800007FFA93C45010 c10.dll!c10::MessageLogger::~MessageLogger [<unknown file> @ <unknown line number>]
00007FFA93C4631300007FFA93C462E0 c10.dll!c10::ThrowEnforceNotMet [<unknown file> @ <unknown line number>]
00007FFA93C3812300007FFA93C37F70 c10.dll!c10::alloc_cpu [<unknown file> @ <unknown line number>]
00007FFA93BF1A2C00007FFA93BF19F0 c10.dll!c10::DefaultCPUAllocator::allocate [<unknow

✓ step 18 → ' grab'  saved as 'step-18-grab'


Precomputation completed in 84.80s
Found 2714656 active features
Phase 1: Running forward pass
Forward pass completed in 1115.62s
Phase 2: Building input vectors
Using 1 salient logits with cumulative probability 1.0000
Will include 1024 of 2714656 feature nodes
Input vectors built in 5.39s
Phase 3: Computing logit attributions
Logit attributions completed in 216.13s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [1:50:34<00:00,  6.48s/it]
Feature attributions completed in 6634.64s
Attribution completed in 8057.32s


✓ step 19 → ' it'  saved as 'step-19-it'


# 2. attribute graph visualization

In [9]:
import json
from pathlib import Path

graph_dir = Path("./graph_files/gemma-3-1b")

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"gemma-2-2b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

step-00-He.json: 10 layers → ['gemma-2-2b/15-gemmascope-transcoder-16k', 'gemma-2-2b/16-gemmascope-transcoder-16k', 'gemma-2-2b/18-gemmascope-transcoder-16k']...
step-01-saw.json: 11 layers → ['gemma-2-2b/15-gemmascope-transcoder-16k', 'gemma-2-2b/16-gemmascope-transcoder-16k', 'gemma-2-2b/17-gemmascope-transcoder-16k']...
step-02-a.json: 11 layers → ['gemma-2-2b/15-gemmascope-transcoder-16k', 'gemma-2-2b/16-gemmascope-transcoder-16k', 'gemma-2-2b/17-gemmascope-transcoder-16k']...
step-03-carrot.json: 10 layers → ['gemma-2-2b/16-gemmascope-transcoder-16k', 'gemma-2-2b/17-gemmascope-transcoder-16k', 'gemma-2-2b/18-gemmascope-transcoder-16k']...
step-04-and.json: 14 layers → ['gemma-2-2b/10-gemmascope-transcoder-16k', 'gemma-2-2b/13-gemmascope-transcoder-16k', 'gemma-2-2b/14-gemmascope-transcoder-16k']...
step-05-had.json: 10 layers → ['gemma-2-2b/15-gemmascope-transcoder-16k', 'gemma-2-2b/16-gemmascope-transcoder-16k', 'gemma-2-2b/18-gemmascope-transcoder-16k']...
step-06-to.json: 11 la

In [10]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(Path("./graph_files/gemma-3-1b").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/gemma-3-1b/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [ ]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/gemma-3-1b/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

## 2.1 feature analysis based on circuit

In [ ]:
# inspect one graph file
data = json.loads(list(Path("./graph_files/gemma-3-1b/").glob("step-*.json"))[0].read_text())
for n in data['nodes'][:10]:
    print(n)

In [ ]:
import json
from pathlib import Path

top_features = set()

for json_path in sorted(Path("./graph_files/gemma-3-1b/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    nodes = [n for n in data['nodes'] 
             if not n['is_target_logit'] 
             and n['influence'] is not None
             and n['feature_type'] == 'cross layer transcoder']  # ← only transcoder nodes
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:10]
    for node in nodes:
        parts = node['node_id'].split('_')
        layer = parts[0]   # '13', '14', '15' etc
        feat = parts[1]    # '1005', '2100' etc
        
        if layer.isdigit():
            top_features.add((int(layer), int(feat)))

In [ ]:
print(f"Unique top features: {len(top_features)}")
print(list(top_features)[:5])

In [ ]:
from collections import Counter

layer_counts = Counter(layer for layer, feat in top_features)
for layer in sorted(layer_counts):
    print(f"Layer {layer}: {layer_counts[layer]} features")

# 3. intervention
based on intervention_demo

In [ ]:
from collections import namedtuple
from functools import partial

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

In [ ]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [ ]:
supernode_features = [
    #Feature(layer=17,pos=-1,feature_idx=15160),
    #Feature(layer=17,pos=-1,feature_idx=5452),
    Feature(layer=16,pos=-1,feature_idx=11787),
    #Feature(layer=16,pos=-1,feature_idx=5798),
    #Feature(layer=17,pos=-1,feature_idx=15724),
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [ ]:
s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

In [ ]:
with torch.inference_mode():
    original_logits, _  = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

In [ ]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

# suppress both candidate features at the newline position (-1)
intervention_tuples = [
    (5, -1, 6651, 0.0),
    (17, -1, 15160, 0.0),
    (17, -1, 5452, 0.0),
    (16, -1, 11787, 0.0),
    (16, -1, 5798, 0.0),
    (17, -1, 15724, 0.0),
]

print("Generating with original features...")
pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=True)[0]]

print("\nGenerating with interventions...")
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=True)[0]]

display_generations_comparison(s, pre, post)